In [0]:
for t in ["dim_customer", "dim_card", "dim_merchant", "dim_date", "fact_transactions"]:
    print(f"{t}: {spark.table(t).count():,} rows")

dim_customer: 25,000 rows
dim_card: 32,457 rows
dim_merchant: 500 rows
dim_date: 1,326 rows
fact_transactions: 250,000 rows


### Fraud Overview

#### How much fraud do we have overall?

In [0]:
spark.sql("""
    SELECT 
        COUNT(*) AS total_transactions,
        SUM(fraud_flag) AS fraud_transactions,
        ROUND(100.0 * SUM(fraud_flag) / COUNT(*), 2) AS fraud_rate_pct,
        ROUND(SUM(CASE WHEN fraud_flag = 1 THEN transaction_amount ELSE 0 END), 2) AS total_fraud_amount,
        ROUND(AVG(CASE WHEN fraud_flag = 1 THEN transaction_amount END), 2) AS avg_fraud_amount
    FROM fact_transactions
""").show()

+------------------+------------------+--------------+------------------+----------------+
|total_transactions|fraud_transactions|fraud_rate_pct|total_fraud_amount|avg_fraud_amount|
+------------------+------------------+--------------+------------------+----------------+
|            250000|             13473|          5.39|     1052200974.24|        78097.01|
+------------------+------------------+--------------+------------------+----------------+



### Fraud by Merchant Category

####  Which merchant categories are riskiest?

In [0]:
df = spark.sql("""
    SELECT 
        merchant_category,
        COUNT(*) AS total_txns,
        SUM(fraud_flag) AS fraud_txns,
        ROUND(100.0 * SUM(fraud_flag) / COUNT(*), 2) AS fraud_rate_pct,
        ROUND(SUM(CASE WHEN fraud_flag = 1 THEN transaction_amount ELSE 0 END), 2) AS fraud_amount
    FROM fact_transactions
    GROUP BY merchant_category
    ORDER BY fraud_amount DESC
""")
display(df)

merchant_category,total_txns,fraud_txns,fraud_rate_pct,fraud_amount
Hospital,17870,973,5.44,113030930.15
Hotel,20946,1143,5.46,103152054.64
Airline,18620,992,5.33,100529171.63
Travel,19434,1062,5.46,90003221.22
Financial Services,16564,873,5.27,87979050.09
Electronics,14103,754,5.35,81511103.33
Education,13536,681,5.03,74953506.48
Fuel,19475,1062,5.45,64969152.21
Ecommerce,17545,953,5.43,63279151.25
Fashion,19525,1014,5.19,58345116.70


### Fraud by Customer State

#### Which states have the highest fraud activity?

In [0]:
df2 = spark.sql("""
    SELECT 
        c.state,
        COUNT(*) AS total_txns,
        SUM(f.fraud_flag) AS fraud_txns,
        ROUND(100.0 * SUM(f.fraud_flag) / COUNT(*), 2) AS fraud_rate_pct
    FROM fact_transactions f
    JOIN dim_customer c ON f.customer_key = c.customer_key
    GROUP BY c.state
    ORDER BY fraud_txns DESC
""")
display(df2)

state,total_txns,fraud_txns,fraud_rate_pct
Uttar Pradesh,17077,1016,5.95
Telangana,16866,972,5.76
Gujarat,16510,946,5.73
Kerala,17358,933,5.38
Haryana,16556,921,5.56
Odisha,16572,916,5.53
Karnataka,17275,907,5.25
Bihar,16458,898,5.46
West Bengal,16960,886,5.22
Punjab,16547,882,5.33


### Fraud by Card Type

#### Are credit cards more fraud-prone than debit?

In [0]:
df3 = spark.sql("""
    SELECT 
        ca.card_type,
        COUNT(*) AS total_txns,
        SUM(f.fraud_flag) AS fraud_txns,
        ROUND(100.0 * SUM(f.fraud_flag) / COUNT(*), 2) AS fraud_rate_pct
    FROM fact_transactions f
    JOIN dim_card ca ON f.card_key = ca.card_key
    GROUP BY ca.card_type
    ORDER BY fraud_rate_pct DESC
""")
display(df3)

card_type,total_txns,fraud_txns,fraud_rate_pct
Platinum,46415,2587,5.57
Gold,76927,4145,5.39
Classic,56998,3065,5.38
Silver,69660,3676,5.28


###  Fraud by Transaction Channel

#### Which channel has the most fraud — ATM, Mobile App, POS, Online?

In [0]:
df4 = spark.sql("""
    SELECT 
        transaction_channel,
        COUNT(*) AS total_txns,
        SUM(fraud_flag) AS fraud_txns,
        ROUND(100.0 * SUM(fraud_flag) / COUNT(*), 2) AS fraud_rate_pct,
        ROUND(SUM(CASE WHEN fraud_flag = 1 THEN transaction_amount ELSE 0 END), 2) AS fraud_amount
    FROM fact_transactions
    GROUP BY transaction_channel
    ORDER BY fraud_rate_pct DESC
""")
display(df4)

transaction_channel,total_txns,fraud_txns,fraud_rate_pct,fraud_amount
Mobile App,64991,3613,5.56,294026294.92
Online Web,24823,1371,5.52,106344798.79
ATM,39902,2122,5.32,163895152.45
POS,120284,6367,5.29,487934728.08


###  Time-of-Day Fraud Patterns

#### What hours are fraudsters most active?

In [0]:
df5 = spark.sql("""
    SELECT 
        HOUR(transaction_time) AS hour_of_day,
        COUNT(*) AS total_txns,
        SUM(fraud_flag) AS fraud_txns,
        ROUND(100.0 * SUM(fraud_flag) / COUNT(*), 2) AS fraud_rate_pct
    FROM fact_transactions
    GROUP BY HOUR(transaction_time)
    ORDER BY fraud_rate_pct DESC
    LIMIT 10
""")
display(df5)

hour_of_day,total_txns,fraud_txns,fraud_rate_pct
0,12059,700,5.80
2,10278,585,5.69
6,10457,592,5.66
13,10360,584,5.64
4,10377,580,5.59
12,10321,576,5.58
20,10253,571,5.57
11,10408,575,5.52
21,10296,562,5.46
5,10459,570,5.45


### International vs Domestic Fraud

####  Is international fraud more frequent than domestic?

In [0]:
df6 = spark.sql("""
    SELECT 
        CASE WHEN is_international = 1 THEN 'International' ELSE 'Domestic' END AS txn_type,
        COUNT(*) AS total_txns,
        SUM(fraud_flag) AS fraud_txns,
        ROUND(100.0 * SUM(fraud_flag) / COUNT(*), 2) AS fraud_rate_pct,
        ROUND(AVG(transaction_amount), 2) AS avg_amount
    FROM fact_transactions
    GROUP BY is_international
""")
display(df6)

txn_type,total_txns,fraud_txns,fraud_rate_pct,avg_amount
International,12572,2476,19.69,39657.79
Domestic,237428,10997,4.63,15387.00


### Fraud Reasons Breakdown

#### What are the top reasons flagged for fraud?

In [0]:
df7 = spark.sql("""
    SELECT 
        fraud_reason,
        COUNT(*) AS occurrences,
        ROUND(SUM(transaction_amount), 2) AS total_amount
    FROM fact_transactions
    WHERE fraud_flag = 1 AND fraud_reason IS NOT NULL
    GROUP BY fraud_reason
    ORDER BY occurrences DESC
""")
display(df7)

fraud_reason,occurrences,total_amount
Transaction Attempt on Expired Card,2470,19382441.39
High Risk Merchant Category Spike,1758,81757475.77
Unusual Cross-Border / International Transaction,1721,75670491.56
Unusual High Transaction Amount vs Credit Limit,1707,772020041.54
Rapid Velocity & Multiple Failed Auth Attempts,1702,17927997.69
Late Night Geographical & Velocity Anomaly,1676,66478143.06
Transaction Attempt on Blocked Card,1591,12551227.54
Transaction Attempt on Lost Card,848,6413155.69


### Top 10 Fraudulent Merchants

#### Which merchants have the highest fraud amount?

In [0]:
df8 = spark.sql("""
    SELECT 
        m.merchant_name,
        m.city,
        m.merchant_risk_level,
        COUNT(*) AS fraud_txns,
        ROUND(SUM(f.transaction_amount), 2) AS total_fraud_amount
    FROM fact_transactions f
    JOIN dim_merchant m ON f.merchant_key = m.merchant_key
    WHERE f.fraud_flag = 1
    GROUP BY m.merchant_name, m.city, m.merchant_risk_level
    ORDER BY total_fraud_amount DESC
    LIMIT 10
""")
display(df8)

merchant_name,city,merchant_risk_level,fraud_txns,total_fraud_amount
Max Healthcare,Durgapur,Low,54,10056429.84
Croma,Sohna,Medium,35,5773702.95
Samsung SmartCafe,Gandhinagar,High,35,5770326.08
MakeMyTrip Hotels,Bhavnagar,Medium,36,5755896.34
Fortis Healthcare,Dibrugarh,Low,37,5625513.11
MakeMyTrip Hotels,Dibrugarh,High,32,5465504.81
Udemy,Gwalior,High,17,5103476.72
Myntra,Guwahati,Low,59,5097258.28
Reliance Digital,Aligarh,Medium,33,5090168.00
Vijay Sales,Faridabad,Medium,34,5069765.02


### Month-over-Month Fraud Trend

#### Is fraud increasing or decreasing over time?

In [0]:
df9 = spark.sql("""
    SELECT 
        d.year,
        d.month,
        d.month_name,
        COUNT(*) AS total_txns,
        SUM(f.fraud_flag) AS fraud_txns,
        ROUND(100.0 * SUM(f.fraud_flag) / COUNT(*), 2) AS fraud_rate_pct
    FROM fact_transactions f
    JOIN dim_date d ON f.date_key = d.date_key
    GROUP BY d.year, d.month, d.month_name
    ORDER BY d.year, d.month
""")
display(df9)

year,month,month_name,total_txns,fraud_txns,fraud_rate_pct
2023,1,January,6027,304,5.04
2023,2,February,5564,297,5.34
2023,3,March,6372,360,5.65
2023,4,April,6235,350,5.61
2023,5,May,6349,333,5.24
2023,6,June,6179,322,5.21
2023,7,July,6414,333,5.19
2023,8,August,6237,348,5.58
2023,9,September,5638,299,5.30
2023,10,October,5690,302,5.31
